<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# **Introduction to Meridian Demo**

Welcome to the Meridian end-to-end demo. This simplified demo showcases the fundamental functionalities and basic usage of the library, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
</ol>


Note that this notebook skips all of the exploratory data analysis and preprocessing steps. It assumes that you have completed these tasks before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

<a name="install"></a>
## Step 0: Install

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [1]:
# # Install meridian: from PyPI @ latest release
# !pip install --upgrade google-meridian[colab,and-cuda]

# # Install meridian: from PyPI @ specific version
# # !pip install google-meridian[colab,and-cuda]==1.1.1

# # Install meridian: from GitHub @HEAD
# # !pip install --upgrade "google-meridian[colab,and-cuda] @ git+https://github.com/google/meridian.git@main"

In [2]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 25.8 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv) as follows.

1\. Read the data into a Pandas DataFrame.

In [9]:
# df = pd.read_csv(
#     "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv"
# )

df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/national_all_channels.csv"
)

2\. Create a DataFrameInputDataBuilder instance.

In [10]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

3\. Offer the components to the builder. Note that the components may be offered all at once or piecewise.

In [11]:
builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    # .with_population(df)
    .with_controls(
        df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    )
)

channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
)

4. If your data includes organic media or non-media treatments, you can add them using `with_organic_media` and `with_non_media_treatments` methods. For the definition of each variable, see
[Collect and organize your data](https://developers.google.com/meridian/docs/user-guide/collect-data)

In [12]:
builder = builder.with_non_media_treatments(
    df, non_media_treatment_cols=['Promo']
).with_organic_media(
    df,
    organic_media_cols=['Organic_channel0_impression'],
    organic_media_channels=['Organic_channel0'],
)

5. Finally, build the InputData.

In [13]:
data = builder.build()

In [14]:
data.media

<xarray.DataArray 'media' (geo: 1, media_time: 156, media_channel: 5)> Size: 6kB
array([[[ 44340630,  10537596,   6501889,  66738506,  53026011],
        [ 47391940,  30857600,   7966320,  79000018,  27592214],
        [ 28960185,  23144749,   4637739,  68442555,  20057461],
        [ 38126016,  10667077,   3029637,  70601091,  37246623],
        [ 46890794,  17925961,  10486840,  81842660,  56259530],
        [ 31496378,  19099596,   3713252,  58861970,  26886668],
        [ 25772939,   4734654,   7271164,  68725846,  37615253],
        [ 35435485,  17853348,   8084840,  73628817,  40915556],
        [ 28273144,  27489089,   6237490,  64406105,  56582012],
        [ 48102115,  18370050,  10814751,  54768193,  31935062],
        [ 19006405,  14689528,  16603730,  80575635,  34549374],
        [ 11884095,  22511541,   8715033,  64287874,  43085333],
        [ 20987050,  10287056,   1303124,  39119396,  26850814],
        [ 32721647,  22413235,         0,  51764884,  22663633],
        [ 41690748,   4970780,   6224564,  72170268,  43131568],
        [  7851189,  26950502,   7470685,  96670719,  37692690],
        [ 24068825,  34650278,   9308505,  76949051,  26845617],
        [ 33899092,   7764352,   1153694,  45123891,  12497175],
        [ 29650250,  20997265,   3306149,  57501044,  36937240],
        [ 30096729,  27484493,   9678321,  95433081,  35712590],
...
        [ 58141486,  52366813,  29052657, 119448062,  51475835],
        [ 47930423,  28949513,  16112071, 100549485,  65612761],
        [ 20401702,  30927605,    974522,  54608381,  39696931],
        [ 49075722,  10613048,  30234773,  93406375,  61576311],
        [ 45906183,  34528362,  29004263,  95047720,  62275392],
        [ 49832323,  20111088,  30364481,  99078370,  55679447],
        [ 38771745,  21241838,   9088634,  84043242,  38857711],
        [ 54643380,  20617793,  28553055, 101646966,  72264612],
        [ 35139539,  14349455,   5799921,  58158620,  25735245],
        [ 51199673,  39464748,   6652452,  67235543,  39509435],
        [ 39662913,  31410884,  16425557,  84986626,  61962255],
        [ 27105781,  25245751,  20662827,  78205700,  37197498],
        [ 48429249,  31854098,  26927336,  73980025,  60474143],
        [ 46113938,  44771420,  38020008,  78193832,  49502334],
        [ 53163003,  10788168,   7997572, 100553951,  39580963],
        [ 15690802,  33468942,   2132232,  75586335,  18067454],
        [ 36158974,  17591779,  13006993,  79997178,  28549297],
        [ 33297423,  20483606,  13339158,  86878440,  59321564],
        [ 40819684,  31527719,   3524627,  51605579,  42044958],
        [ 36349940,  10502860,  11270574,  83076800,  27057902]]])
Coordinates:
  * media_time     (media_time) object 1kB '2021-01-25' ... '2024-01-15'
  * media_channel  (media_channel) object 40B 'Channel0' ... 'Channel4'
  * geo            (geo) <U12 48B 'national_geo'

Note that the simulated data here does not contain reach and frequency. We recommend including reach and frequency data whenever they are available. For information about the advantages of utilizing reach and frequency, see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/#:~:text=By%20incorporating%20R%26F%20into%20MMM,based%20on%20optimal%20frequency%20recommendations.). For code snippet for loading reach and frequency data, see [Load geo-level data with reach and frequency](https://developers.google.com/meridian/docs/user-guide/load-geo-data-with-rf)

The documentation provides guidance for instances where reach and frequency data is accessible for specific channels. Additionally, for information about how to load other data types and formats, including data with reach and frequency, see [Supported data types and formats](https://developers.google.com/meridian/docs/user-guide/supported-data-types-formats).

<a name="configure-model"></a>
## Step 2: Configure the model

Meridian uses Bayesian framework and Markov Chain Monte Carlo (MCMC) algorithms to sample from the posterior distribution.

1\. Inititalize the `Meridian` class by passing the loaded data and the customized model specification. One advantage of Meridian lies in its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). In this particular example, the ROI priors for all media channels are identical, with each being represented as Lognormal(0.2, 0.9).

In [15]:
# roi_mu = 0.2  # Mu for ROI prior for each media channel.
# roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
# prior = prior_distribution.PriorDistribution(
#     roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
# )
model_spec = spec.ModelSpec()

mmm = model.Meridian(input_data=data, model_spec=model_spec)

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
I0000 00:00:1756330245.132186 1062315 service.cc:148] XLA service 0x12c2728e0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756330245.132215 1062315 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1756330245.138911 1062315 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [16]:
mmm.media_tensors.media.shape

TensorShape([1, 156, 5])

2\. Use the `sample_prior()` and `sample_posterior()` methods to obtain samples from the prior and posterior distributions of model parameters. If you are using the T4 GPU runtime this step may take about 10 minutes for the provided data set.

In [9]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

For more information about configuring the parameters and using a customized model specification, such as setting different ROI priors for each media channel, see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

<a name="model-diagnostics"></a>
## Step 3: Run model diagnostics

After the model is built, you must assess convergence, debug the model if needed, and then assess the model fit.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [10]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [11]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 4: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [12]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [13]:
from google.colab import drive

drive.mount('/content/drive')

In [14]:
filepath = '/content/drive/MyDrive'
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

Here is a preview of the two-page output based on the simulated data:

In [15]:
IPython.display.HTML(filename='/content/drive/MyDrive/summary_output.html')

For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 5: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [16]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [17]:
filepath = '/content/drive/MyDrive'
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [18]:
IPython.display.HTML(filename='/content/drive/MyDrive/optimization_output.html')

For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).

<a name="save-model"></a>
## Step 6: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [19]:
file_path = '/content/drive/MyDrive/saved_mmm.pkl'
model.save_mmm(mmm, file_path)

Run the following codes to load the saved model:

In [20]:
mmm = model.load_mmm(file_path)